# Procesamiento ETL 2023 - Estadísticas Vitales

Este notebook se encarga de importar, limpiar, manejar nulos, normalizar y guardar de forma eficiente (en formato Parquet) los datasets correspondientes al año 2023 para:
- Nacimientos
- Muertes Fetales
- Muertes No Fetales

In [ ]:
import os
import sys
import pandas as pd

# Agregar la carpeta principal al path para poder importar módulos propios
sys.path.append(os.path.abspath('..'))

from etl.cleaning import *
from etl.nulls import *
from etl.normalization import *

# Asegurarnos de que directorios de salida existan
os.makedirs('../data/processed/nacimientos', exist_ok=True)
os.makedirs('../data/processed/fetales', exist_ok=True)
os.makedirs('../data/processed/no_fetales', exist_ok=True)

## 1. Nacimientos 2023
### Carga del archivo

In [ ]:
rutaDatos = '../data/raw/nac2023.csv'
try:
    df_nac = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_nac.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.DataFrame'>
RangeIndex: 515549 entries, 0 to 515548
Data columns (total 39 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   COD_DPTO        515549 non-null  int64  
 1   COD_MUNIC       515549 non-null  int64  
 2   AREANAC         515549 non-null  int64  
 3   SIT_PARTO       515549 non-null  int64  
 4   OTRO_SIT        1037 non-null    str    
 5   SEXO            515549 non-null  int64  
 6   PESO_NAC        515549 non-null  int64  
 7   TALLA_NAC       515549 non-null  int64  
 8   ANO             515549 non-null  int64  
 9   MES             515549 non-null  int64  
 10  ATEN_PAR        515549 non-null  int64  
 11  T_GES           515549 non-null  int64  
 12  T_GES_AGRU_CIE  515549 non-null  int64  
 13  NUMCONSUL       515549 non-null  int64  
 14  TIPO_PARTO      515549 non-null  int64  
 15  MUL_PARTO       515549 non-null  int64  
 16  APGAR1          515549 non-null  int64  
 17 

### Eliminación de duplicados

In [ ]:
encontrar_duplicados(df_nac)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 515549
Filas duplicadas (incluyendo repetidas): 2657
Duplicados únicos a eliminar: 1568


,COD_DPTO,COD_MUNIC,AREANAC,SIT_PARTO,OTRO_SIT,SEXO,PESO_NAC,TALLA_NAC,ANO,MES,...,N_HIJOSV,FECHA_NACM,N_EMB,SEG_SOCIAL,IDCLASADMI,EDAD_PADRE,NIV_EDUP,ULTCURPAD,PROFESION,TIPOFORMULARIO
267,5,1,1,1,NaN,1,4,4,2023,9,...,1,NaN,1,2,2.0,23,3,8,1.0,1
3045,5,1,1,1,NaN,1,4,4,2023,9,...,1,NaN,1,2,2.0,23,3,8,1.0,1
4351,13,1,1,1,NaN,1,4,4,2023,4,...,2,NaN,2,2,2.0,30,2,5,1.0,1
6074,73,411,1,1,NaN,1,5,4,2023,6,...,3,04/08/2014,2,2,2.0,42,5,11,1.0,1
6075,73,411,1,1,NaN,1,5,4,2023,6,...,3,04/08/2014,2,2,2.0,42,5,11,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
513480,8,1,1,1,NaN,2,5,4,2023,10,...,1,NaN,1,2,2.0,25,4,11,1.0,1
513663,76,1,1,1,NaN,1,7,5,2023,1,...,1,NaN,1,1,1.0,25,4,11,1.0,1
514647,47,1,1,1,NaN,1,5,4,2023,1,...,3,27/09/2018,2,1,1.0,29,9,5,1.0,1
514648,47,1,1,1,NaN,1,5,4,2023,1,...,3,27/09/2018,2,1,1.0,29,9,5,1.0,1


In [ ]:
df_nac = eliminar_duplicados(df_nac)

===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 1568
Registros eliminados: 1568
Total final: 513981


### Eliminación de columnas que no son necesarias para el estudio

In [ ]:
cols_a_borrar_nac = parsear_columnas_eliminar("../colEliminar.md", "Nacimientos")
df_nac = eliminar_columnas(df_nac, cols_a_borrar_nac)

Borradas 9 columnas de la lista especificada.


In [ ]:
encontrar_duplicados(df_nac)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 513981
Filas duplicadas (incluyendo repetidas): 686
Duplicados únicos a eliminar: 343


,COD_DPTO,COD_MUNIC,AREANAC,SIT_PARTO,SEXO,PESO_NAC,TALLA_NAC,ANO,MES,ATEN_PAR,...,CODPRES,CODPTORE,CODMUNRE,AREA_RES,N_HIJOSV,SEG_SOCIAL,IDCLASADMI,EDAD_PADRE,NIV_EDUP,PROFESION
149,11,1,1,1,2,6,5,2023,11,1,...,170.0,11.0,1.0,1.0,1,1,1.0,20,4,1.0
2033,50,1,1,1,1,4,4,2023,3,1,...,170.0,50.0,1.0,1.0,2,5,NaN,22,4,1.0
3867,19,1,1,1,2,5,4,2023,11,1,...,170.0,19.0,532.0,3.0,4,2,2.0,44,99,1.0
3868,19,1,1,1,2,5,4,2023,11,1,...,170.0,19.0,532.0,3.0,4,2,2.0,44,99,1.0
4347,5,1,1,1,2,4,4,2023,4,1,...,170.0,5.0,1.0,1.0,3,2,2.0,43,99,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
507773,20,1,1,1,1,5,4,2023,10,1,...,170.0,20.0,178.0,1.0,3,2,2.0,38,7,1.0
508979,23,1,1,1,1,5,4,2023,2,1,...,170.0,23.0,1.0,1.0,3,2,2.0,25,8,1.0
511043,76,1,1,1,2,6,5,2023,8,1,...,170.0,76.0,1.0,1.0,1,1,1.0,23,4,1.0
511814,68,1,1,1,1,4,4,2023,8,1,...,170.0,68.0,547.0,3.0,2,2,2.0,22,5,1.0


### Estandarización de Nombres de las columnas

In [ ]:
df_nac = estandarizar_nombres_columnas(df_nac)

### Manejo de nulos 
Al revisar los nulos, mantenemos el criterio del 2020: variables geográficas sin datos se eliminan. `idclasadmi` se imputa con 'DESCONOCIDO'.

In [ ]:
resumen_nulos = analizar_nulos(df_nac)


===== ANÁLISIS DE NULOS =====
Total de registros: 513981
Columnas con nulos: 6



,nulos,porcentaje (%)
idclasadmi,23624,4.596279
codptore,6161,1.198682
area_res,6161,1.198682
codmunre,6161,1.198682
profesion,3322,0.646327
codpres,3001,0.583874


In [ ]:
# 2. eliminar nulos geográficos
df_nac = df_nac.dropna(subset=["codptore", "codmunre", "area_res"])

# 3. imputar otros nulos
estrategia = {
    "idclasadmi": {"metodo": "fill", "valor": "DESCONOCIDO"}
}
df_nac = manejar_nulos(df_nac, estrategia)

### Ajustar Tipos de Datos y Guardado

In [ ]:
df_nac = ajustar_tipos_datos(df_nac)
""" out_nac = '../data/processed/nacimientos/nacimientos_2023.parquet'
df_nac.to_parquet(out_nac, index=False, engine='pyarrow')
print(f"Guardado Nacimientos 2023 en: {out_nac}") """


Guardado Nacimientos 2023 en: ../data/processed/nacimientos/nacimientos_2023.parquet


## 2. Muertes Fetales 2023
### Carga del archivo

In [ ]:
rutaDatos = '../data/raw/fetal2023.csv'
try:
    df_fet = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_fet.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.DataFrame'>
RangeIndex: 24862 entries, 0 to 24861
Data columns (total 46 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   COD_DPTO        24862 non-null  int64  
 1   COD_MUNIC       24862 non-null  int64  
 2   A_DEFUN         24862 non-null  int64  
 3   SIT_DEFUN       24862 non-null  int64  
 4   OTRSITIODE      53 non-null     str    
 5   TIPO_DEFUN      24862 non-null  int64  
 6   ANO             24862 non-null  int64  
 7   MES             24862 non-null  int64  
 8   HORA            11673 non-null  float64
 9   MINUTOS         11673 non-null  float64
 10  SEXO            24862 non-null  int64  
 11  CODPRES         24783 non-null  float64
 12  CODPTORE        24560 non-null  float64
 13  CODMUNRE        24560 non-null  float64
 14  AREA_RES        24562 non-null  float64
 15  SEG_SOCIAL      24862 non-null  int64  
 16  IDADMISALUD     23413 non-null  float64
 17  P_PMAN_IRIS     2486

### Eliminación de duplicados

In [ ]:
encontrar_duplicados(df_fet)
df_fet = eliminar_duplicados(df_fet)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 24862
Filas duplicadas (incluyendo repetidas): 280
Duplicados únicos a eliminar: 143
===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 143
Registros eliminados: 143
Total final: 24719


### Eliminación de columnas

In [ ]:
cols_a_borrar_fet = parsear_columnas_eliminar("../colEliminar.md", "Fetales")
df_fet = eliminar_columnas(df_fet, cols_a_borrar_fet)

Borradas 21 columnas de la lista especificada.


### Estandarizar Nombres de Columnas

In [ ]:
df_fet = estandarizar_nombres_columnas(df_fet)

### Manejo de nulos

In [ ]:
resumen_nulos = analizar_nulos(df_fet)
display(resumen_nulos)

df_fet = df_fet.dropna(subset=["codptore", "codmunre", "area_res"])

estrategia = {
    "idadmisalud": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_fet = manejar_nulos(df_fet, estrategia)

===== ANÁLISIS DE NULOS =====
Total de registros: 24719
Columnas con nulos: 4



,nulos,porcentaje (%)
idadmisalud,1443,5.837615
codmunre,301,1.217687
codptore,301,1.217687
area_res,299,1.209596


,nulos,porcentaje (%)
idadmisalud,1443,5.837615
codmunre,301,1.217687
codptore,301,1.217687
area_res,299,1.209596


### Ajustar Tipos de Datos y Guardado

In [ ]:
df_fet = ajustar_tipos_datos(df_fet)
""" out_fet = '../data/processed/fetales/fetales_2023.parquet'
df_fet.to_parquet(out_fet, index=False, engine='pyarrow')
print(f"Guardado Muertes Fetales 2023 en: {out_fet}") """


Guardado Muertes Fetales 2023 en: ../data/processed/fetales/fetales_2023.parquet


## 3. Muertes No Fetales 2023
### Carga del archivo

In [ ]:
rutaDatos = '../data/raw/nofetal2023.csv'
try:
    df_nofet = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_nofet.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.DataFrame'>
RangeIndex: 268411 entries, 0 to 268410
Data columns (total 59 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   COD_DPTO        268411 non-null  int64  
 1   COD_MUNIC       268411 non-null  int64  
 2   A_DEFUN         268411 non-null  int64  
 3   SIT_DEFUN       268411 non-null  int64  
 4   OTRSITIODE      7693 non-null    str    
 5   TIPO_DEFUN      268411 non-null  int64  
 6   ANO             268411 non-null  int64  
 7   MES             268411 non-null  int64  
 8   HORA            260855 non-null  float64
 9   MINUTOS         260855 non-null  float64
 10  SEXO            268411 non-null  int64  
 11  EST_CIVIL       268411 non-null  int64  
 12  GRU_ED1         268411 non-null  int64  
 13  GRU_ED2         268411 non-null  int64  
 14  NIVEL_EDU       268411 non-null  int64  
 15  ULTCURFAL       268411 non-null  int64  
 16  MUERTEPORO      261356 non-null  float64
 17 

### Eliminar duplicados

In [ ]:
encontrar_duplicados(df_nofet)
df_nofet = eliminar_duplicados(df_nofet)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 268411
Filas duplicadas (incluyendo repetidas): 399
Duplicados únicos a eliminar: 224
===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 224
Registros eliminados: 224
Total final: 268187


### Eliminación de columnas que no nos sirven para el estudio 

In [ ]:
cols_a_borrar_nofet = parsear_columnas_eliminar("../colEliminar.md", "No fetales")
df_nofet = eliminar_columnas(df_nofet, cols_a_borrar_nofet)

columnas_eliminar = [
    "simuertepo", "t_ges", "niv_edum", "n_hijosm", "tipo_emb", "t_parto", 
    "est_civm", "peso_nac", "edad_madre", "n_hijosv", "emb_mes", 
    "emb_sem", "emb_fal"
]
print(f"Eliminando columnas adicionales vacías...")
# Solo eliminamos las que realmente existan en el dataframe despues de borrar cols_a_borrar_nofet
cols_to_drop = [c for c in columnas_eliminar if c in df_nofet.columns]
df_nofet = df_nofet.drop(columns=cols_to_drop)

Borradas 24 columnas de la lista especificada.
Eliminando columnas adicionales vacías...


### Estandarizar nombre de columnas

In [ ]:
df_nofet = estandarizar_nombres_columnas(df_nofet)

### Manejo de nulos

In [ ]:
resumen_nulos = analizar_nulos(df_nofet)

===== ANÁLISIS DE NULOS =====
Total de registros: 268187
Columnas con nulos: 19



,nulos,porcentaje (%)
simuertepo,268049,99.948543
n_hijosm,262580,97.909295
est_civm,262580,97.909295
n_hijosv,262580,97.909295
t_parto,262580,97.909295
peso_nac,262580,97.909295
edad_madre,262580,97.909295
tipo_emb,262580,97.909295
t_ges,262580,97.909295
niv_edum,262580,97.909295


In [ ]:
resumen_nulos = analizar_nulos(df_nofet)
display(resumen_nulos)

df_nofet = df_nofet.dropna(subset=["codptore", "codmunre", "area_res"])

estrategia = {
    "ocupacion": {"metodo": "fill", "valor": "DESCONOCIDO"},
    "muerteporo": {"metodo": "fill_mode"},
    "idadmisalud": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_nofet = manejar_nulos(df_nofet, estrategia)

===== ANÁLISIS DE NULOS =====
Total de registros: 268187
Columnas con nulos: 19



,nulos,porcentaje (%)
simuertepo,268049,99.948543
n_hijosm,262580,97.909295
est_civm,262580,97.909295
n_hijosv,262580,97.909295
t_parto,262580,97.909295
peso_nac,262580,97.909295
edad_madre,262580,97.909295
tipo_emb,262580,97.909295
t_ges,262580,97.909295
niv_edum,262580,97.909295


,nulos,porcentaje (%)
simuertepo,268049,99.948543
n_hijosm,262580,97.909295
est_civm,262580,97.909295
n_hijosv,262580,97.909295
t_parto,262580,97.909295
peso_nac,262580,97.909295
edad_madre,262580,97.909295
tipo_emb,262580,97.909295
t_ges,262580,97.909295
niv_edum,262580,97.909295


### Ajustar Tipos de Datos y Guardado

In [ ]:
df_nofet = ajustar_tipos_datos(df_nofet)
""" out_nofet = '../data/processed/no_fetales/nofetales_2023.parquet'
df_nofet.to_parquet(out_nofet, index=False, engine='pyarrow')
print(f"Guardado Muertes No Fetales 2023 en: {out_nofet}") """


Guardado Muertes No Fetales 2023 en: ../data/processed/no_fetales/nofetales_2023.parquet
